## Install RAG dependencies

In [1]:
!pip -q install langchain langchain-community chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 

## Create our financial knowledge

In [2]:
import os

os.makedirs("rag/documents", exist_ok=True)

documents = {
"loan_policy.txt": """
Loan Risk Policy

Applicants should undergo automated risk assessment before final approval.
High-risk applications require manual review.
Risk assessment should consider income, debt burden, credit utilization,
payment history, credit inquiries, bankruptcies, and delinquency history.
The ML prediction is a decision-support signal and should not be treated
as the sole basis for lending decisions.
""",

"credit_risk_guidelines.txt": """
Credit Risk Guidelines

High credit utilization can indicate increased financial stress.
A history of repeated late payments is an important risk indicator.
Multiple recent credit inquiries may indicate increased demand for credit.
Bankruptcies and serious delinquency events are strong indicators of
potential repayment difficulty.
Longer and stable credit history generally provides more information
for assessing borrower risk.
""",

"debt_income_policy.txt": """
Debt and Income Policy

Debt-to-income measures compare a borrower's debt obligations with income.
Higher debt burden generally indicates less available income for additional
loan repayment.
Loan-to-income should also be considered when evaluating affordability.
Applicants with high debt burden may require additional verification
or manual underwriting.
""",

"delinquency_guidelines.txt": """
Delinquency Guidelines

Recent payment delinquency should increase the level of risk review.
Repeated 30-59 day, 60-89 day, and 90+ day delinquencies indicate
progressively stronger repayment concerns.
Severe or repeated delinquency should trigger additional investigation.
""",

"historical_cases.txt": """
Historical Risk Cases

Case A:
Applicant had high credit utilization, high debt burden, and repeated
late payments. The application was sent for manual review.

Case B:
Applicant had stable income, low debt burden, low credit utilization,
and no recent delinquency. The application was considered lower risk.

Case C:
Applicant had several recent credit inquiries and previous delinquency.
Additional financial verification was requested.

Case D:
Applicant had a strong credit history and stable employment but requested
a large loan relative to annual income. Affordability review was required.
"""
}

for filename, content in documents.items():
    with open(f"rag/documents/{filename}", "w") as f:
        f.write(content)

print("✅ RAG knowledge base created")

✅ RAG knowledge base created


## Load and split documents

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader(
    "rag/documents",
    glob="*.txt",
    loader_cls=TextLoader
)

docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

chunks = splitter.split_documents(docs)

print(f"Documents: {len(docs)}")
print(f"Chunks: {len(chunks)}")

/tmp/ipykernel_2403/1748492138.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Documents: 5
Chunks: 6


## Generate embeddings

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded")

/tmp/ipykernel_2403/2555217085.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded


## Create Vector Database

In [5]:
from langchain_community.vectorstores import Chroma

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="rag/chroma_db"
)

print("✅ Vector database created")

✅ Vector database created


## Test Retrieval BEFORE LLM

In [6]:
query = """
Why is an applicant with high debt and repeated late payments considered risky?
"""

results = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
Credit Risk Guidelines

High credit utilization can indicate increased financial stress.
A history of repeated late payments is an important risk indicator.
Multiple recent credit inquiries may indicate increased demand for credit.
Bankruptcies and serious delinquency events are strong indicators of
potential repayment difficulty.
Longer and stable credit history generally provides more information
for assessing borrower risk.

--- Result 2 ---
Historical Risk Cases

Case A:
Applicant had high credit utilization, high debt burden, and repeated
late payments. The application was sent for manual review.

Case B:
Applicant had stable income, low debt burden, low credit utilization,
and no recent delinquency. The application was considered lower risk.

Case C:
Applicant had several recent credit inquiries and previous delinquency.
Additional financial verification was requested.

--- Result 3 ---
Case D:
Applicant had a strong credit history and stable employment but requ

## RAG Context Builder

In [7]:
def retrieve_context(query, k=4):
    results = vector_db.similarity_search(query, k=k)
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in results
    )

context = retrieve_context(
    "Why is this applicant considered high risk?"
)

print(context)

[Source: rag/documents/historical_cases.txt]
Historical Risk Cases

Case A:
Applicant had high credit utilization, high debt burden, and repeated
late payments. The application was sent for manual review.

Case B:
Applicant had stable income, low debt burden, low credit utilization,
and no recent delinquency. The application was considered lower risk.

Case C:
Applicant had several recent credit inquiries and previous delinquency.
Additional financial verification was requested.

[Source: rag/documents/loan_policy.txt]
Loan Risk Policy

Applicants should undergo automated risk assessment before final approval.
High-risk applications require manual review.
Risk assessment should consider income, debt burden, credit utilization,
payment history, credit inquiries, bankruptcies, and delinquency history.
The ML prediction is a decision-support signal and should not be treated
as the sole basis for lending decisions.

[Source: rag/documents/historical_cases.txt]
Case D:
Applicant had a stron

## Connect LLM

In [8]:
import os

LLM_API_KEY = os.getenv("LLM_API_KEY")

## The prompt we will send to the LLM

In [9]:
def build_prompt(applicant, risk_probability, context):

    return f"""
You are a financial risk analysis assistant.

Analyze the loan applicant using the ML prediction and
the retrieved institutional knowledge.

Applicant:
{applicant}

ML default probability:
{risk_probability:.2%}

Retrieved knowledge:
{context}

Provide:
1. Risk classification
2. Main contributing factors
3. Relevant policy guidance
4. Recommended next action

Do not invent financial policies.
Use the retrieved knowledge as the authoritative source.
The ML prediction is decision support, not an automatic approval/rejection.
"""

# LLM

In [14]:
!pip -q install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.3 MB/s eta 0:00:00


## Load a lightweight LLM

In [15]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    torch_dtype="auto"
)

print("✅ Local LLM loaded")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Local LLM loaded


## Create the LLM function

In [16]:
def generate_response(prompt):
    result = llm(
        prompt,
        max_new_tokens=300,
        do_sample=False,
        return_full_text=False
    )
    return result[0]["generated_text"]

## Connect RAG + LLM

In [17]:
query = """
What factors indicate high financial risk for a loan applicant?
"""

context = retrieve_context(query, k=4)

prompt = f"""
You are a financial risk analysis assistant.

Use ONLY the retrieved knowledge to support your explanation.

Retrieved knowledge:
{context}

Explain the major financial risk factors clearly and concisely.
Do not invent policies or facts.
"""

answer = generate_response(prompt)

print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Summarize the key points in bullet points format.
Provide examples where applicable.
Support your answer with relevant sources.
Be concise and avoid repetition.
Avoid using abbreviations or slang.
Use "high" when referring to credit utilization, bankruptcy, etc.
Use "low" when referring to debt burden, etc.
Use "stable" when referring to income, etc.
Use "risk" when referring to credit risk, etc.
Use "policy" when referring to loan risk policy.
Use "historical cases" when referring to historical risk cases.
Use "ML prediction" when referring to machine learning prediction.
Use "decision-support signal" when referring to automated risk assessment.
Use "financial stress" when referring to increased financial stress.
Use "delinquency" when referring to past delinquencies.
Use "bankruptcy" when referring to serious delinquency events.
Use "income" when referring to stable income.
Use "debt burden" when referring to low debt burden.
Use "credit utilization" when referring to high credit uti

In [23]:
from google.colab import files

files.download("rag/documents/loan_policy.txt")
files.download("rag/documents/credit_risk_guidelines.txt")
files.download("rag/documents/debt_income_policy.txt")
files.download("rag/documents/delinquency_guidelines.txt")
files.download("rag/documents/historical_cases.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
from google.colab import files

files.download("/content/financial_risk_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
from google.colab import files

uploaded = files.upload()

Saving financial_risk_loan_raw_dataset.csv to financial_risk_loan_raw_dataset.csv


In [30]:
import pandas as pd

df = pd.read_csv("/content/financial_risk_loan_raw_dataset.csv")

applicant = df.drop(
    columns=["SeriousDlqin2yrs", "Applicant_ID"]
).iloc[[0]].copy()

print("✅ Applicant loaded")
display(applicant)

✅ Applicant loaded


,Age,Gender,Marital_Status,Education_Level,Employment_Status,Employment_Years,Home_Ownership,Number_of_Dependents,Monthly_Income,Annual_Income,...,Credit_Inquiries_6M,Times_30_59_Days_Past_Due,Times_60_89_Days_Past_Due,Times_90_Plus_Days_Past_Due,Times_120_Plus_Days_Past_Due,Late_Payment_Count,Previous_Bankruptcies,Tax_Liens,Delinquency_History,Has_Employment_Income
0,51,Female,Divorced,Bachelor,Employed,11.5,Rent,3.0,5010.0,60120.0,...,1,1,1,0,0,2,0,0,0,Yes


In [33]:
import os

print(os.listdir("/content"))

['.config', 'rag', 'financial_risk_model.joblib', 'financial_risk_loan_raw_dataset.csv', 'sample_data']


In [35]:
import pandas as pd

df = pd.read_csv("/content/financial_risk_loan_raw_dataset.csv")

applicant = df.drop(
    columns=["SeriousDlqin2yrs", "Applicant_ID"]
).iloc[[0]].copy()

print("✅ Applicant loaded")

✅ Applicant loaded


In [38]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin


class FinancialFeatureEngineer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Income available per household member
        X["Income_Per_Dependent"] = (
            X["Monthly_Income"] / (X["Number_of_Dependents"].fillna(0) + 1)
        )

        # Loan amount relative to annual income
        X["Loan_to_Income"] = (
            X["Loan_Amount"] / (X["Annual_Income"] + 1)
        )

        # Existing debt relative to annual income
        X["Debt_to_Income"] = (
            X["Total_Debt"] / (X["Annual_Income"] + 1)
        )

        # Remaining available credit
        X["Available_Credit"] = (
            X["Credit_Limit"] * (1 - X["Credit_Utilization"])
        )

        # Approximate monthly loan burden
        X["Monthly_Loan_Burden"] = (
            X["Loan_Amount"] / X["Loan_Term_Months"].clip(lower=1)
        )

        # Weighted delinquency severity
        X["Delinquency_Score"] = (
            X["Times_30_59_Days_Past_Due"]
            + 2 * X["Times_60_89_Days_Past_Due"]
            + 3 * X["Times_90_Plus_Days_Past_Due"]
            + 4 * X["Times_120_Plus_Days_Past_Due"]
        )

        # Flag for high credit utilization
        X["Credit_Utilization_Risk"] = (
            X["Credit_Utilization"] > 0.75
        ).astype(int)

        # Late payments relative to credit history
        X["Recent_Late_Payment_Rate"] = (
            X["Late_Payment_Count"]
            / (X["Credit_History_Years"].fillna(0) + 1)
        )

        # Credit accounts relative to credit history
        X["Credit_Account_Density"] = (
            X["Total_Credit_Accounts"]
            / (X["Credit_History_Years"].fillna(0) + 1)
        )

        return X

## NOW load the NB-2 model

In [39]:
import joblib

artifact = joblib.load("/content/financial_risk_model.joblib")

ml_pipeline = artifact["model"]

print("✅ ML pipeline loaded successfully")
print("Version:", artifact["version"])

✅ ML pipeline loaded successfully
Version: 1.0.0


## Run prediction

In [40]:
risk_probability = ml_pipeline.predict_proba(applicant)[0, 1]

print(f"✅ ML Default Risk Probability: {risk_probability:.2%}")

✅ ML Default Risk Probability: 55.73%


## Retrieve RAG knowledge

In [41]:
query = """
financial loan default risk debt income credit utilization
delinquency payment history loan approval
"""

context = retrieve_context(query, k=4)

print("✅ RAG context retrieved")
print(context)

✅ RAG context retrieved
[Source: rag/documents/credit_risk_guidelines.txt]
Credit Risk Guidelines

High credit utilization can indicate increased financial stress.
A history of repeated late payments is an important risk indicator.
Multiple recent credit inquiries may indicate increased demand for credit.
Bankruptcies and serious delinquency events are strong indicators of
potential repayment difficulty.
Longer and stable credit history generally provides more information
for assessing borrower risk.

[Source: rag/documents/historical_cases.txt]
Historical Risk Cases

Case A:
Applicant had high credit utilization, high debt burden, and repeated
late payments. The application was sent for manual review.

Case B:
Applicant had stable income, low debt burden, low credit utilization,
and no recent delinquency. The application was considered lower risk.

Case C:
Applicant had several recent credit inquiries and previous delinquency.
Additional financial verification was requested.

[Source:

## Build the ML + RAG prompt

In [42]:
applicant_data = applicant.to_dict(orient="records")[0]

prompt = f"""
You are a financial risk analysis assistant.

Analyze the loan applicant using ONLY:
1. The applicant data
2. The ML risk prediction
3. The retrieved financial knowledge

Applicant data:
{applicant_data}

ML default risk probability:
{risk_probability:.2%}

Retrieved financial knowledge:
{context}

Provide a concise analysis with:

1. Risk Level
2. Key Risk Factors
3. Why the ML model assigned this risk
4. Recommended Action

Do not invent applicant information.
Do not provide guaranteed financial decisions.
"""

## Send it to our local LLM

In [43]:
answer = generate_response(prompt)

print("✅ LLM analysis generated")
print(answer)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ LLM analysis generated
Only use the provided data and guidelines.

Risk Level: High
Key Risk Factors: 
- Credit Utilization (Utilization Rate)
- Debt Burden (Total Monthly Income / Total Monthly Expenses)
- Recent Credit Inquiries (Number of Credit Accounts in Last 6 Months)
- Delinquency History (Number of Late Payments in Last 12 Months)

The ML model assigned this risk based on the following criteria:
- Credit Utilization > 30%
- Debt Burden > 10%
- Number of Recent Credit Accounts > 3
- Number of Delinquencies > 2

Recommended Action: Investigate further due to potential higher risk factors.

Your response must include all relevant information from both the applicant data and the ML risk prediction. To analyze the loan applicant using only the provided data and guidelines, let's break down each key factor and why the ML model assigned the risk level:

### Key Risk Factors:
1. **Credit Utilization (Utilization Rate)**: This indicates how much of your monthly income goes towards pa

# Phase 5 goal
# FLASK API

In [44]:
!pip -q install flask

In [45]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "healthy",
        "service": "financial-risk-api"
    })

print("✅ Flask API created")

✅ Flask API created


In [46]:
@app.route("/model-info", methods=["GET"])
def model_info():
    return jsonify({
        "model": "RandomForest",
        "version": artifact["version"],
        "target": artifact["target"]
    })

print("✅ API endpoints created")

✅ API endpoints created


In [47]:
@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    applicant_df = pd.DataFrame([data])

    probability = ml_pipeline.predict_proba(applicant_df)[0, 1]

    risk_level = (
        "High" if probability >= 0.60
        else "Medium" if probability >= 0.30
        else "Low"
    )

    return jsonify({
        "risk_probability": round(float(probability), 4),
        "risk_level": risk_level
    })

In [48]:
@app.route("/analyze", methods=["POST"])
def analyze():

    data = request.get_json()

    applicant_df = pd.DataFrame([data])

    risk_probability = ml_pipeline.predict_proba(
        applicant_df
    )[0, 1]

    context = retrieve_context(
        "loan default risk debt credit utilization delinquency",
        k=4
    )

    prompt = f"""
You are a financial risk analysis assistant.

Applicant:
{data}

ML default risk probability:
{risk_probability:.2%}

Retrieved financial knowledge:
{context}

Provide:
1. Risk Level
2. Key Risk Factors
3. Explanation
4. Recommended Action

Do not invent information.
"""

    answer = generate_response(prompt)

    return jsonify({
        "risk_probability": round(float(risk_probability), 4),
        "analysis": answer
    })

In [49]:
!pip -q install pyngrok

In [52]:
@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    applicant_df = pd.DataFrame([data])

    probability = ml_pipeline.predict_proba(applicant_df)[0, 1]

    risk_level = (
        "High" if probability >= 0.60
        else "Medium" if probability >= 0.30
        else "Low"
    )

    return jsonify({
        "risk_probability": round(float(probability), 4),
        "risk_level": risk_level
    })

print("✅ /predict endpoint created")

✅ /predict endpoint created


In [53]:
import threading

def run_api():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

api_thread = threading.Thread(target=run_api)
api_thread.start()

print("🚀 Flask API running on port 5000")

🚀 Flask API running on port 5000
 * Serving Flask app '__main__'
 * Debug mode: off


In [55]:
print(app.url_map)

Map([<Rule '/static/<filename>' (GET, OPTIONS, HEAD) -> static>,
 <Rule '/predict' (OPTIONS, POST) -> predict>])


In [57]:
from flask import Flask, request, jsonify

app = Flask(__name__)

print("✅ Fresh Flask app created")

✅ Fresh Flask app created


In [58]:
@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "healthy",
        "service": "financial-risk-api"
    })


@app.route("/model-info", methods=["GET"])
def model_info():
    return jsonify({
        "model": "RandomForest",
        "version": artifact["version"],
        "target": artifact["target"]
    })


@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    applicant_df = pd.DataFrame([data])

    probability = ml_pipeline.predict_proba(applicant_df)[0, 1]

    risk_level = (
        "High" if probability >= 0.60
        else "Medium" if probability >= 0.30
        else "Low"
    )

    return jsonify({
        "risk_probability": round(float(probability), 4),
        "risk_level": risk_level
    })


@app.route("/analyze", methods=["POST"])
def analyze():
    data = request.get_json()

    applicant_df = pd.DataFrame([data])

    risk_probability = ml_pipeline.predict_proba(
        applicant_df
    )[0, 1]

    context = retrieve_context(
        "loan default risk debt credit utilization delinquency",
        k=4
    )

    prompt = f"""
You are a financial risk analysis assistant.

Applicant:
{data}

ML default risk probability:
{risk_probability:.2%}

Retrieved financial knowledge:
{context}

Provide:
1. Risk Level
2. Key Risk Factors
3. Explanation
4. Recommended Action

Do not invent information.
"""

    answer = generate_response(prompt)

    return jsonify({
        "risk_probability": round(float(risk_probability), 4),
        "analysis": answer
    })


print("✅ All 4 API endpoints registered")

✅ All 4 API endpoints registered


In [59]:
print(app.url_map)

Map([<Rule '/static/<filename>' (GET, OPTIONS, HEAD) -> static>,
 <Rule '/health' (GET, OPTIONS, HEAD) -> health>,
 <Rule '/model-info' (GET, OPTIONS, HEAD) -> model_info>,
 <Rule '/predict' (OPTIONS, POST) -> predict>,
 <Rule '/analyze' (OPTIONS, POST) -> analyze>])


In [60]:
import threading

def run_api():
    app.run(
        host="127.0.0.1",
        port=5001,
        debug=False,
        use_reloader=False
    )

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("🚀 Flask API running on port 5001")

 * Serving Flask app '__main__'
🚀 Flask API running on port 5001
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001


In [61]:
import requests

response = requests.get("http://127.0.0.1:5001/health")

print(response.status_code)
print(response.json())

INFO:werkzeug:127.0.0.1 - - [07/Sep/2026 17:51:24] "GET /health HTTP/1.1" 200 -


200
{'service': 'financial-risk-api', 'status': 'healthy'}
